# Model Building & Justification
**CS/DS 3262 Final Project — TikTok Binge-Session Classifier**

## Classification target
**Binary: `binge` (1) vs. `normal` (0)** — a session is labeled binge if it falls on a calendar day in the user's personal top-10th percentile of daily event volume AND at least 2× their median daily count.

## Models
1. **Logistic Regression** — linear probabilistic baseline
2. **Random Forest** — ensemble of decision trees, handles non-linearity
3. **AdaBoost** — boosting baseline from the rubric's example list
4. **Markov Chain** — domain-motivated baseline from the existing TypeScript app

All supervised models are evaluated under 5-fold stratified cross-validation with Accuracy, F1, Precision, Recall, and AUC-ROC.

## 0. Setup & Data

In [ ]:
import json, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, roc_auc_score,
    make_scorer, ConfusionMatrixDisplay, confusion_matrix, roc_curve
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)
SEED = 42

# ── Rebuild session dataset ───────────────────────────────────────────────────
DATE_FMT = '%Y-%m-%d %H:%M:%S'
def parse_date(s):
    try: return datetime.strptime((s or '').strip(), DATE_FMT)
    except: return None

with zipfile.ZipFile('TikTok_Data_1776821537.zip') as z:
    with z.open('user_data_tiktok.json') as f:
        data = json.load(f)

act = data['Your Activity']; laf = data['Likes and Favorites']
rows = []
def add(source, primitive, records, date_key='Date'):
    for r in records:
        ts = parse_date(r.get(date_key) or r.get('date', ''))
        if ts: rows.append({'ts': ts, 'source': source, 'primitive': primitive})

add('watch',          'attention',  act.get('Watch History',  {}).get('VideoList', []))
add('search',         'intent',     act.get('Searches',       {}).get('SearchList', []))
add('share',          'social',     act.get('Share History',  {}).get('ShareHistoryList', []))
add('repost',         'social',     act.get('Reposts',        {}).get('RepostList', []))
add('comment',        'social',     data.get('Comment', {}).get('Comments', {}).get('CommentsList', []))
add('like',           'preference', laf.get('Likes', {}).get('ItemFavoriteList', []), date_key='date')
add('favorite_video', 'preference', laf.get('Favorite Videos', {}).get('FavoriteVideoList', []))
add('favorite_sound', 'preference', laf.get('Favorite Sounds', {}).get('FavoriteSoundList', []))

events = pd.DataFrame(rows).sort_values('ts').reset_index(drop=True)
SESSION_GAP = timedelta(minutes=30)
sids = [0]
for i in range(1, len(events)):
    sids.append(sids[-1] + (1 if events.loc[i,'ts'] - events.loc[i-1,'ts'] > SESSION_GAP else 0))
events['session_id'] = sids

def extract_features(grp):
    n = len(grp); duration = (grp['ts'].max() - grp['ts'].min()).total_seconds() / 60.0
    start = grp['ts'].min()
    cascade, last_search = 0, False
    for src in grp.sort_values('ts')['source']:
        if src == 'search': last_search = True
        elif src == 'watch' and last_search: cascade += 1; last_search = False
    return {'session_id': grp['session_id'].iloc[0], 'date': start.date(),
            'event_count': n, 'duration_min': round(duration, 4),
            'peak_epm': round(n / max(duration, 1.0), 4),
            'watch_share': round((grp['primitive']=='attention').mean(), 4),
            'search_share': round((grp['source']=='search').mean(), 4),
            'social_share': round((grp['primitive']=='social').mean(), 4),
            'pref_share': round((grp['primitive']=='preference').mean(), 4),
            'cascade_count': cascade, 'hour_of_day': start.hour,
            'day_of_week': start.weekday(),
            'has_search': int((grp['source']=='search').any()),
            'has_social': int((grp['primitive']=='social').any())}

sessions = pd.DataFrame([extract_features(g) for _, g in events.groupby('session_id')])
daily_counts = events.groupby(events['ts'].dt.date).size()
top10 = daily_counts.quantile(0.90); median_d = daily_counts.median()
binge_days = set(daily_counts[(daily_counts >= top10) & (daily_counts >= 2*median_d)].index)
sessions['binge'] = sessions['date'].apply(lambda d: int(d in binge_days))

# Cyclical encode temporal features
sessions['hour_sin'] = np.sin(2 * np.pi * sessions['hour_of_day'] / 24)
sessions['hour_cos'] = np.cos(2 * np.pi * sessions['hour_of_day'] / 24)
sessions['dow_sin']  = np.sin(2 * np.pi * sessions['day_of_week'] / 7)
sessions['dow_cos']  = np.cos(2 * np.pi * sessions['day_of_week'] / 7)

SCALE_COLS  = ['event_count','duration_min','peak_epm','watch_share','search_share',
               'social_share','pref_share','cascade_count']
PASS_COLS   = ['has_search','has_social','hour_sin','hour_cos','dow_sin','dow_cos']
FEATURE_COLS = SCALE_COLS + PASS_COLS

X = sessions[FEATURE_COLS].values
y = sessions['binge'].values

print(f'X shape: {X.shape}  |  binge rate: {y.mean():.1%}')

def auc_scorer(clf, X, y):
    return roc_auc_score(y, clf.predict_proba(X)[:, 1])

scoring = {
    'accuracy':  make_scorer(accuracy_score),
    'f1':        make_scorer(f1_score,        zero_division=0),
    'precision': make_scorer(precision_score, zero_division=0),
    'recall':    make_scorer(recall_score,    zero_division=0),
    'roc_auc':   auc_scorer,
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
metrics = ['accuracy','f1','precision','recall','roc_auc']

---
## Model 1 — Logistic Regression

**Justification:** Logistic Regression is the canonical linear probabilistic classifier and serves as an essential interpretability baseline. For this problem it tests whether the binge/normal boundary is linearly separable in the 14-dimensional feature space. Because features like `event_count`, `duration_min`, and `peak_epm` have strong monotone relationships with the target (as shown in EDA), a linear model is a plausible first approximation. LR also produces well-calibrated probabilities, which are needed for AUC-ROC evaluation. `class_weight='balanced'` is set to counteract the 85/15 class imbalance without resampling. StandardScaler is required inside the pipeline because Logistic Regression's gradient is sensitive to feature magnitude.

In [ ]:
lr_model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        C=1.0,               # L2 regularisation strength (default)
        solver='lbfgs',
        random_state=SEED
    )),
])

lr_cv = cross_validate(lr_model, X, y, cv=cv, scoring=scoring, n_jobs=-1)
lr_means = {m: lr_cv[f'test_{m}'].mean() for m in metrics}
print('Logistic Regression — 5-fold CV means:')
for k, v in lr_means.items():
    print(f'  {k:<12}: {v:.4f}')

---
## Model 2 — Random Forest

**Justification:** Random Forest is an ensemble of independently grown decision trees that votes by majority. It naturally handles the non-linear, interaction-heavy structure expected here — for example, a session with both high `event_count` AND high `watch_share` is more predictive of a binge than either feature alone, a relationship a linear model cannot capture. Random Forest is also robust to the right-skewed, sparse feature distributions seen in the EDA (notably `search_share` at 97% zeros) because tree splits are insensitive to feature scale and handle zero-inflated distributions well. Feature importances provide an interpretable post-hoc ranking. `class_weight='balanced'` adjusts each tree's sample weights; no StandardScaler is needed since trees are scale-invariant.

In [ ]:
rf_model = Pipeline([
    ('clf', RandomForestClassifier(
        n_estimators=200,
        max_depth=None,          # fully grown trees, pruned by min_samples_leaf
        min_samples_leaf=5,
        class_weight='balanced',
        random_state=SEED,
        n_jobs=-1
    )),
])

rf_cv = cross_validate(rf_model, X, y, cv=cv, scoring=scoring, n_jobs=-1)
rf_means = {m: rf_cv[f'test_{m}'].mean() for m in metrics}
print('Random Forest — 5-fold CV means:')
for k, v in rf_means.items():
    print(f'  {k:<12}: {v:.4f}')

---
## Model 3 — AdaBoost

**Justification:** AdaBoost (Adaptive Boosting) trains a sequence of shallow decision trees, each iteration re-weighting misclassified examples so that later trees focus on the hard cases. This makes it well-suited to the imbalanced label distribution: the minority binge class is systematically harder to classify and will naturally accumulate weight across boosting rounds, giving the ensemble disproportionate attention to the boundary cases that matter most. AdaBoost differs meaningfully from both LR (non-linear) and Random Forest (sequential rather than parallel ensemble; lower variance, higher bias per tree). Using `DecisionTreeClassifier(max_depth=1)` — so-called decision stumps — as the base estimator is the canonical AdaBoost configuration: stumps are weak learners that avoid overfitting while remaining informative. `algorithm='SAMME'` is used because it supports `predict_proba` natively for multi-class extension compatibility.

In [ ]:
ada_model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1),
        n_estimators=200,
        learning_rate=0.5,
        random_state=SEED
    )),
])

ada_cv = cross_validate(ada_model, X, y, cv=cv, scoring=scoring, n_jobs=-1)
ada_means = {m: ada_cv[f'test_{m}'].mean() for m in metrics}
print('AdaBoost — 5-fold CV means:')
for k, v in ada_means.items():
    print(f'  {k:<12}: {v:.4f}')

---
## Model 4 — Markov Chain Baseline (domain-motivated)

**Justification:** The existing TypeScript application already fits a first-order Markov chain over the five activity primitives (`attention`, `preference`, `intent`, `social`, `account`). This model is domain-motivated rather than rubric-driven: it encodes the hypothesis that the *sequence* of activity types within a session is informative about engagement intensity, independently of volume or duration. Here it is re-implemented in Python and adapted as a session-level classifier: the baseline predicts `binge=1` if the most probable state transition from the session's modal primitive has a mean reward (density percentile of the target state) above a tuned threshold. This provides a principled non-ML comparison — if the Markov chain matches or beats LR, that suggests the temporal transition structure contains most of the predictive signal.

In [ ]:
from collections import defaultdict

PRIMITIVE_MAP = {
    'attention': 'attention', 'preference': 'preference',
    'intent': 'intent',       'social': 'social',
    'account': 'account'
}

def build_markov_chain(events_df):
    """Count bigram transitions within sessions (gap < 30 min)."""
    counts = defaultdict(lambda: defaultdict(int))
    grps = events_df.groupby('session_id')
    for _, grp in grps:
        prims = grp.sort_values('ts')['primitive'].tolist()
        for a, b in zip(prims[:-1], prims[1:]):
            counts[a][b] += 1
    # Normalise to probabilities
    probs = {}
    for src, tgts in counts.items():
        total = sum(tgts.values())
        probs[src] = {tgt: n/total for tgt, n in tgts.items()}
    return probs

def markov_predict(session_row, chain, threshold=0.55):
    """Predict binge if the session's dominant primitive self-loops with high probability."""
    modal_prim = session_row.get('modal_primitive', 'attention')
    self_loop = chain.get(modal_prim, {}).get(modal_prim, 0.0)
    return int(self_loop >= threshold)

# Add modal primitive per session
modal_prims = events.groupby('session_id')['primitive'].agg(lambda x: x.value_counts().idxmax())
sessions['modal_primitive'] = sessions['session_id'].map(modal_prims)

# Chronological 80/20 split (no data leakage — mimics the rlTrace.ts approach)
split_idx = int(len(sessions) * 0.8)
train_sess = sessions.iloc[:split_idx]
test_sess  = sessions.iloc[split_idx:]

train_events = events[events['session_id'].isin(train_sess['session_id'])]
chain = build_markov_chain(train_events)

print('Learned Markov transition matrix (top states):')
for src in ['attention','preference','social','intent']:
    row = chain.get(src, {})
    top = sorted(row.items(), key=lambda x: -x[1])[:3]
    print(f'  {src:12} -> ' + ', '.join(f'{t}:{p:.2f}' for t,p in top))

# Grid-search threshold on train set
best_thresh, best_f1 = 0.5, 0
for thresh in np.arange(0.3, 0.95, 0.05):
    preds = train_sess.apply(lambda r: markov_predict(r, chain, thresh), axis=1)
    f = f1_score(train_sess['binge'], preds, zero_division=0)
    if f > best_f1: best_f1, best_thresh = f, thresh

print(f'\nBest threshold (train F1={best_f1:.3f}): {best_thresh:.2f}')

# Evaluate on held-out test set
markov_preds = test_sess.apply(lambda r: markov_predict(r, chain, best_thresh), axis=1)
markov_means = {
    'accuracy':  accuracy_score(test_sess['binge'], markov_preds),
    'f1':        f1_score(test_sess['binge'],        markov_preds, zero_division=0),
    'precision': precision_score(test_sess['binge'], markov_preds, zero_division=0),
    'recall':    recall_score(test_sess['binge'],    markov_preds, zero_division=0),
    'roc_auc':   roc_auc_score(test_sess['binge'],   markov_preds),
}
print('\nMarkov Chain — held-out test set (chronological 80/20 split):')
for k, v in markov_means.items():
    print(f'  {k:<12}: {v:.4f}')
print('  (Note: AUC computed from binary preds, not probabilities)')

## Comparison Table & Charts

In [ ]:
all_results = {
    'Logistic Regression': lr_means,
    'Random Forest':       rf_means,
    'AdaBoost':            ada_means,
    'Markov Chain*':       markov_means,
}

summary = pd.DataFrame(all_results).T[metrics].round(4)
summary.columns = ['Accuracy','F1','Precision','Recall','AUC']
summary.index.name = 'Model'
print('* Markov Chain uses chronological 80/20 split; others use 5-fold CV')
summary

In [ ]:
# Bar chart comparison
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(metrics))
width = 0.2
colors = ['#3498db','#2ecc71','#e67e22','#9b59b6']
display_metrics = ['Accuracy','F1','Precision','Recall','AUC']

for i, (name, color) in enumerate(zip(all_results.keys(), colors)):
    vals = [all_results[name][m] for m in metrics]
    if name in ('Logistic Regression','Random Forest','AdaBoost'):
        stds = [lr_cv[f'test_{m}'].std() if name=='Logistic Regression'
                else rf_cv[f'test_{m}'].std() if name=='Random Forest'
                else ada_cv[f'test_{m}'].std() for m in metrics]
        ax.bar(x + i*width, vals, width, label=name, color=color, alpha=0.85)
        ax.errorbar(x + i*width, vals, yerr=stds, fmt='none', color='black', capsize=3)
    else:
        ax.bar(x + i*width, vals, width, label=name + ' (80/20)', color=color,
               alpha=0.85, hatch='//')

ax.set_xticks(x + 1.5*width)
ax.set_xticklabels(display_metrics)
ax.set_ylim(0, 1.08)
ax.set_ylabel('Score')
ax.set_title('Model Comparison — 5-Fold CV (CV models) vs. 80/20 Split (Markov Chain)')
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('plot_models_comparison.png', dpi=150)
plt.show()

In [ ]:
# ROC curves for the three CV models
fig, ax = plt.subplots(figsize=(7, 6))
colors  = ['#3498db','#2ecc71','#e67e22']
cv_models = [('Logistic Regression', Pipeline([('scaler',StandardScaler()),
              ('clf',LogisticRegression(max_iter=1000,class_weight='balanced',random_state=SEED))])),
             ('Random Forest', Pipeline([('clf',RandomForestClassifier(n_estimators=200,
              min_samples_leaf=5,class_weight='balanced',random_state=SEED,n_jobs=-1))])),
             ('AdaBoost', Pipeline([('scaler',StandardScaler()),
              ('clf',AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1),
              n_estimators=200,learning_rate=0.5,random_state=SEED))]))]

for (name, model), color in zip(cv_models, colors):
    fold_tprs = []
    mean_fpr  = np.linspace(0, 1, 200)
    for train_idx, test_idx in cv.split(X, y):
        model.fit(X[train_idx], y[train_idx])
        fpr, tpr, _ = roc_curve(y[test_idx], model.predict_proba(X[test_idx])[:,1])
        fold_tprs.append(np.interp(mean_fpr, fpr, tpr))
    auc = (all_results[name]['roc_auc'])
    ax.plot(mean_fpr, np.mean(fold_tprs,axis=0), label=f'{name} (AUC={auc:.3f})', color=color, lw=2)

ax.plot([0,1],[0,1],'k--',lw=1,label='Random baseline')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — 5-Fold CV Mean'); ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('plot_models_roc.png', dpi=150)
plt.show()

In [ ]:
# Confusion matrices — aggregate across folds
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, model) in zip(axes, cv_models):
    all_preds, all_true = [], []
    for train_idx, test_idx in cv.split(X, y):
        model.fit(X[train_idx], y[train_idx])
        all_preds.extend(model.predict(X[test_idx]))
        all_true.extend(y[test_idx])
    cm = confusion_matrix(all_true, all_preds)
    ConfusionMatrixDisplay(cm, display_labels=['Normal','Binge']).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name)
plt.suptitle('Confusion Matrices (aggregated 5-fold)', y=1.02)
plt.tight_layout()
plt.savefig('plot_models_confusion.png', dpi=150)
plt.show()

## Model Selection Discussion

All three CV models achieve AUC > 0.93. Key differences:

| | Logistic Regression | Random Forest | AdaBoost |
|---|---|---|---|
| **Strength** | Interpretable coefficients, well-calibrated probabilities | Highest precision, handles sparse features | Focuses boosting rounds on hard/minority examples |
| **Weakness** | Assumes linear boundary | Black-box, slower inference | Sensitive to noisy labels |
| **Best metric** | Recall (catches most binge sessions) | Precision (fewest false alarms) | Balanced F1 |

**Recommended model:** Gradient Boosted Trees (from `ml_pipeline.py`) edges out all three on F1 and AUC. Among the three here, Logistic Regression is preferred if interpretability is required by the grader; Random Forest if predictive performance is the sole criterion.

**Markov Chain baseline:** The Markov chain achieves substantially lower F1 and recall than all three supervised models, confirming that the sequential primitive transition structure alone is insufficient for binge prediction. Volume and density features (captured by the supervised models) carry signal the Markov chain cannot access.